In [24]:
import torch
from transformers import AutoModel, EsmTokenizer, AutoModelForMaskedLM
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime
import time
from src.architecture import Denoiser
from src.diffusion import Diffusion
from src.util import LengthSampler

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ESM_MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
EMBEDDING_DIM = 320
MAX_LEN = 70
TIMESTEPS = 1000 
now=datetime.now()
now = now.strftime("%m%d%H%M")
OUTPUT_FILE = f"Sequence/{now}.fasta"
!which pip
WEIGHTS_PATH = "trained_models_new/epoch_100.pth" 
DECODER_PATH = "trained_models_cfg/decoder_final.pth"


COND_CONFIGS = [
    {'label_idx': 0, 'scale': 0.4}, # AMP
    {'label_idx': 1, 'scale': 1.5},  # Non-HEMOLYTIC
    {'label_idx': 2, 'scale': 1.2}
]
# COND_CONFIGS = [
#     {'label_idx': 1, 'scale': 6}, 

# ]
# COND_CONFIGS = [
#     # {'label_idx': 0, 'scale': 0.5}, 
#     {'label_idx': 1, 'scale': 2},  
#     # {'label_idx': 2, 'scale': 5}
# ]
# COND_CONFIGS = [
#     {'label_idx': 0, 'scale': 0.9}, 
#     {'label_idx': 1, 'scale': 1.8},  
#     {'label_idx': 2, 'scale': 0.4}
# ]
def main():
    print("--- Generation ---")
    
    print("Loading ESM...")
    tokenizer = EsmTokenizer.from_pretrained(ESM_MODEL_NAME)
    esm_model = AutoModel.from_pretrained(ESM_MODEL_NAME)
    
    print("Loading Denoiser...")
    denoiser = Denoiser(esm_model, input_dim=EMBEDDING_DIM, dropout_prob=0.0).to(DEVICE)
    
    if not os.path.exists(WEIGHTS_PATH):
        print(f"Error: Weights {WEIGHTS_PATH} not found.")
        return
    denoiser.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
    print(denoiser)
    diffusion = Diffusion(denoiser, timesteps=TIMESTEPS).to(DEVICE)
    
    print("Loading Decoder...")
    decoder = AutoModelForMaskedLM.from_pretrained(ESM_MODEL_NAME).lm_head
    if os.path.exists(DECODER_PATH):
        decoder.load_state_dict(torch.load(DECODER_PATH, map_location=DEVICE))
    decoder.to(DEVICE).eval()

    try:
        df = pd.read_csv("data/antimicrobial.csv")
        seqs = df['sequence' if 'sequence' in df.columns else 'SEQUENCE'].dropna().tolist()
        len_sampler = LengthSampler(seqs, max_len=MAX_LEN)
    except:
        print("Using dummy length sampler.")
        len_sampler = LengthSampler(["A"*20]*100, max_len=MAX_LEN)

    NUM_SAMPLES = 1000
    BATCH_SIZE = 128
    valid_seqs = []
    
    pbar = tqdm(total=NUM_SAMPLES)
    batch_cnt = 0
    
    while len(valid_seqs) < NUM_SAMPLES:
        curr_n = min(BATCH_SIZE, (NUM_SAMPLES - len(valid_seqs)) * 2)
        
        latents, mask = diffusion.sample(
            num_samples=curr_n,
            max_len=MAX_LEN,
            embedding_dim=EMBEDDING_DIM,
            length_sampler=len_sampler,
            cond_configs=COND_CONFIGS
        )
        
        # --- debug for tune hyperparameter ---
        if batch_cnt == 0:
            np.save("debug_latent.npy", latents.cpu().numpy())
            np.save("debug_mask.npy", mask.cpu().numpy())
            
            valid_vals = latents[mask.bool()].cpu().numpy()
            print(f"\n[DIAGNOSIS] Valid Latent Max: {valid_vals.max():.4f} (Should be < 15)")
            print(f"[DIAGNOSIS] Valid Latent Min: {valid_vals.min():.4f}")
        # ---------------------------------

        # 解码
        with torch.no_grad():
            logits = decoder(latents)
            temperature = 1.3
            probs = torch.softmax(logits/temperature,dim=-1)
            # pred_ids = torch.argmax(logits, dim=-1) 
            # use this to generate base sequence to check it is learn the method for amp
            #then use temperature to assure variety
            pred_ids = torch.multinomial(probs.view(-1, probs.shape[-1]), 1).view(probs.shape[0], -1).squeeze(-1)
            raw_seqs = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
            
        for i, s in enumerate(raw_seqs):
            real_len = int(mask[i].sum().item())
            clean = s.replace(" ", "")[:real_len]
            #to prevent to generate single animo sequence
            if set(clean).issubset(set("ACDEFGHIKLMNPQRSTVWY")) and len(clean) > 10:
                k_ratio= clean.count('K') /len(clean)
                r_ratio= clean.count('R') /len(clean)
                g_ratio= clean.count('G') /len(clean)
                c_ratio= clean.count('C') /len(clean)
                a_ratio= clean.count('A') /len(clean) 
                l_ratio = clean.count('L')/len(clean)
                m_ratio = clean.count('M')/len(clean)
                # 
                if g_ratio>0.6 or c_ratio >0.6 or a_ratio>0.6 or k_ratio > 0.6 or l_ratio >0.6 or m_ratio>0.6 or r_ratio>0.6:
                    continue
                valid_seqs.append(clean)
                pbar.update(5)
                if len(valid_seqs) >= NUM_SAMPLES: break
        print("Now generate:",len(valid_seqs))

        batch_cnt += 1
            
    pbar.close()
    
    with open(OUTPUT_FILE, "w") as f:
        for i, s in enumerate(valid_seqs):
            f.write(f">gen_{i+1}\n{s}\n")
    print(f"Saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

/home/dlab/software/miniconda3/envs/NHAMP/bin/pip
--- Generation ---
Loading ESM...


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading Denoiser...
Denoiser(
  (time_emb): Sequential(
    (0): SinusoidalPositionEmbeddings()
    (1): Linear(in_features=640, out_features=1280, bias=True)
    (2): SiLU()
    (3): Linear(in_features=1280, out_features=1280, bias=True)
  )
  (label_emb): LabelEmbedder(
    (embedding_table): Embedding(4, 1280)
  )
  (esm_attention_list): ModuleList(
    (0-5): 6 x EsmAttention(
      (self): EsmSelfAttention(
        (query): Linear(in_features=320, out_features=320, bias=True)
        (key): Linear(in_features=320, out_features=320, bias=True)
        (value): Linear(in_features=320, out_features=320, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (rotary_embeddings): RotaryEmbedding()
      )
      (output): EsmSelfOutput(
        (dense): Linear(in_features=320, out_features=320, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
    )
  )
  (time_emb_proj_list): ModuleL

  0%|                                                  | 0/1000 [00:00<?, ?it/s]

Sampling: 0it [00:00, ?it/s]

...

4305it [01:52, 41.58it/s]

Now generate: 946


Sampling: 0it [00:00, ?it/s]

5000it [02:00, 41.36it/s]

Now generate: 1000
Saved to Sequence/01261321.fasta


In [ ]:
%%shell
